# Unit 1 Hands-On: LunarLander-v2 강화학습 실습

이 노트북은 **Hugging Face 딥 강화학습 강좌 Unit 1**의 실습입니다.  
PPO(Proximal Policy Optimization) 알고리즘을 이용해 `LunarLander-v2` 환경에서 에이전트를 훈련하고, Hugging Face Hub에 업로드하는 전 과정을 다룹니다.

---
## 목차
1. 결과 미리보기
2. 환경 설치
3. Google Drive 마운트
4. 가상 디스플레이 설정
5. 라이브러리 임포트
6. 환경 탐색
7. 모델 정의 및 훈련
8. 모델 평가
9. Hugging Face Hub에 업로드

---
## 1. 결과 미리보기

훈련이 완료된 PPO 에이전트가 LunarLander 환경에서 착륙하는 영상입니다.

In [1]:
%%html
<video controls autoplay>
  <source src="https://huggingface.co/sb3/ppo-LunarLander-v2/resolve/main/replay.mp4" type="video/mp4">
</video>

---
## 2. 환경 설치

LunarLander 환경 실행에 필요한 시스템 패키지와 파이썬 패키지를 설치합니다.  
Colab 환경에서 한 번만 실행하면 됩니다.

In [2]:
# Box2D 물리 엔진 의존성 및 빌드 도구 설치
!apt-get update && apt-get install -y \
    libsdl2-dev libsdl2-image-dev libsdl2-mixer-dev libsdl2-ttf-dev \
    libportmidi-dev libfreetype6-dev \
    swig build-essential

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:6 https://cli.github.com/packages stable/main amd64 Packages [356 B]       
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [102 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease   
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,152 kB]
Get:13 http://security.ubuntu.com/ubuntu jam

In [3]:
# v2 호환 패키지 설치 (커뮤니티 해결책 적용)
# requirements-unit1.txt 방식은 setup.py egg-info 오류 발생 → 아래로 대체
!pip install stable-baselines3==2.0.0a5 -q
!pip install swig -q
!pip install gymnasium -q
!pip install box2d-py -q
!pip install huggingface_sb3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.5/177.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 17.3 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.28.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 23.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.5/374.5 kB 5.8 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.3 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0

In [4]:
# swig와 cmake 추가 설치 (Box2D 컴파일에 필요)
!apt install swig cmake

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.0.2-1ubuntu1).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 129 not upgraded.


In [5]:
# 가상 디스플레이 관련 패키지 설치
# Colab은 GUI가 없으므로 가상 디스플레이를 통해 렌더링을 처리합니다.
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg xvfb
!pip3 install pyvirtualdisplay

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The foll

---
## 3. Google Drive 마운트

Colab VM은 **세션 종료 시 모든 파일이 삭제**됩니다.  
Google Drive에 마운트하면 훈련 영상과 모델 파일이 영구적으로 보존됩니다.

```
저장 경로: Google Drive/LunarLander/
           ├── training_videos/        ← 훈련 중 단계별 영상
           └── ppo-LunarLander-v2.zip  ← 최종 모델
```

> 실행하면 Google 계정 인증 팝업이 뜹니다. 허용해주세요.

In [6]:
from google.colab import drive
import os

# Google Drive 마운트 (/content/drive 에 연결)
drive.mount('/content/drive')

# ✏️ Drive 안에 저장할 폴더명을 원하는 대로 변경하세요.
DRIVE_BASE = "/content/drive/MyDrive/LunarLander/Certification"
VIDEO_DIR  = f"{DRIVE_BASE}/training_videos"
MODEL_DIR  = DRIVE_BASE

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"✅ Drive 마운트 완료!")
print(f"   영상 저장 경로 : {VIDEO_DIR}")
print(f"   모델 저장 경로 : {MODEL_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive 마운트 완료!
   영상 저장 경로 : /content/drive/MyDrive/LunarLander/Certification/training_videos
   모델 저장 경로 : /content/drive/MyDrive/LunarLander/Certification


---
## 4. 가상 디스플레이 설정

Colab(서버 환경)에는 모니터가 없기 때문에, 렌더링을 위한 **가상 화면(Virtual Display)**을 만들어야 합니다.

In [7]:
from pyvirtualdisplay import Display

# 보이지 않는(headless) 1400x900 가상 디스플레이 시작
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [8]:
# Hugging Face와 Stable Baselines3 연동 패키지 설치
!pip install huggingface_sb3 -q
!pip install stable-baselines3==2.0.0a5 -q

# Box2D 기반 환경(LunarLander 등) 설치 (v2 호환)
!pip install gymnasium -q
!pip install box2d-py -q

---
## 5. 라이브러리 임포트

| 라이브러리 | 역할 |
|---|---|
| `gymnasium` | 강화학습 환경 (OpenAI Gym의 후속) |
| `stable_baselines3` | PPO 등 강화학습 알고리즘 구현체 |
| `huggingface_sb3` | SB3 모델을 HF Hub에 업로드/다운로드 |
| `huggingface_hub` | HF Hub 로그인 등 인증 처리 |

In [9]:
import imageio
import gymnasium as gym

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback

from IPython.display import Video, display

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---
## 6. 환경 탐색

훈련 전에 환경이 어떻게 생겼는지 먼저 살펴봅니다.

### 5-1. 랜덤 행동으로 환경 테스트

에이전트를 훈련하기 전에, **완전 랜덤한 행동**으로 20스텝을 실행해봅니다.

In [10]:
env = gym.make("LunarLander-v2")
observation, info = env.reset()

for _ in range(20):
    action = env.action_space.sample()  # 랜덤 행동 선택
    print("Action taken:", action)

    # 행동 실행 → 다음 상태, 보상, 종료 여부 반환
    observation, reward, terminated, truncated, info = env.step(action)

    # 에피소드가 끝나면(착륙/충돌/시간초과) 환경 리셋
    if terminated or truncated:
        print("Environment is reset")
        observation, info = env.reset()

env.close()

/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('sphinxcontrib')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-p

Action taken: 3
Action taken: 0
Action taken: 2
Action taken: 3
Action taken: 1
Action taken: 2
Action taken: 3
Action taken: 2
Action taken: 3
Action taken: 1
Action taken: 2
Action taken: 0
Action taken: 2
Action taken: 3
Action taken: 1
Action taken: 3
Action taken: 1
Action taken: 2
Action taken: 1
Action taken: 2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 5-2. 관측 공간(Observation Space) 확인

**관측 공간**: 에이전트가 환경으로부터 받는 정보의 형태입니다.  
LunarLander-v2의 관측값은 **8개의 연속 변수**로 구성됩니다:
- x, y 위치
- x, y 속도
- 각도 및 각속도
- 왼쪽/오른쪽 다리 접지 여부 (0 또는 1)

In [11]:
env = gym.make("LunarLander-v2")
env.reset()

print("===== 관측 공간(Observation Space) =====")
print("Shape:", env.observation_space.shape)       # (8,) → 8개의 실수 값
print("Sample:", env.observation_space.sample())   # 랜덤 샘플 출력

===== 관측 공간(Observation Space) =====
Shape: (8,)
Sample: [-9.979731   -4.3660274   4.4860845   3.5651288  -0.04232434 -2.8494482
  0.84687203  0.9543115 ]


### 5-3. 행동 공간(Action Space) 확인

**행동 공간**: 에이전트가 취할 수 있는 행동의 집합입니다.  
LunarLander-v2는 **이산(discrete) 행동 공간**을 가집니다:

| 행동 번호 | 의미 |
|:---:|---|
| 0 | 아무것도 하지 않음 |
| 1 | 왼쪽 엔진 점화 |
| 2 | 메인(하단) 엔진 점화 |
| 3 | 오른쪽 엔진 점화 |

In [12]:
print("===== 행동 공간(Action Space) =====")
print("Action Space Size:", env.action_space.n)     # 4가지 행동
print("Sample Action:", env.action_space.sample())  # 랜덤 행동 샘플

===== 행동 공간(Action Space) =====
Action Space Size: 4
Sample Action: 0


---
## 7. 모델 정의 및 훈련

### 6-1. 벡터화 환경 생성

**벡터화 환경(Vectorized Environment)**: 여러 환경을 병렬로 동시에 실행하여 데이터 수집 효율을 높입니다.  
`n_envs=16`은 16개의 환경을 병렬 실행하여 훈련을 빠르게 합니다.

In [13]:
# 16개의 LunarLander 환경을 병렬로 생성
env = make_vec_env('LunarLander-v2', n_envs=16)

### 6-2. PPO 모델 정의

**PPO(Proximal Policy Optimization)**: 안정적이고 효율적인 정책 기반 강화학습 알고리즘입니다.

| 하이퍼파라미터 | 값 | 설명 |
|---|---|---|
| `policy` | `'MlpPolicy'` | 다층 퍼셉트론(MLP) 기반 정책 네트워크 |
| `n_steps` | 1024 | 업데이트 전 각 환경에서 수집할 스텝 수 |
| `batch_size` | 64 | 미니배치 크기 |
| `n_epochs` | 4 | 데이터 재사용 횟수 |
| `gamma` | 0.999 | 미래 보상 할인율 (1에 가까울수록 장기 보상 중시) |
| `gae_lambda` | 0.98 | GAE 분산-편향 트레이드오프 조절 |
| `ent_coef` | 0.01 | 탐색을 장려하는 엔트로피 보너스 계수 |

In [14]:
model = PPO(
    policy='MlpPolicy',
    env=env,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    verbose=1  # 훈련 진행 상황 출력
)

Using cpu device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 6-3. 영상 저장 콜백 정의

> ⚠️ VSCode + Jupyter 환경에서 `render_mode="human"`(직접 창 띄우기)은  
> pygame과 Jupyter 커널이 충돌하여 **커널이 죽는 문제**가 있습니다.  
> 대신 `rgb_array`로 프레임을 캡처해 **mp4로 저장**하고 노트북 안에서 바로 재생합니다.

**동작 흐름**:
```
[훈련 환경 x16] ──학습──▶ [모델]
                               │
                  N스텝마다    ▼
          [rgb_array 환경] → 프레임 수집 → mp4 저장 → 노트북에서 재생
```

| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `render_freq` | 30,000 | 몇 스텝마다 영상을 저장할지 |
| `n_eval_episodes` | 1 | 저장할 에피소드 수 |
| `video_dir` | `training_videos` | 영상 저장 폴더 |

In [15]:
class VideoRenderCallback(BaseCallback):
    """
    학습 중 N스텝마다 에피소드를 mp4로 저장하는 콜백.
    - render_mode='human' 대신 'rgb_array'를 사용 → 커널 크래시 없음
    - 저장된 영상은 훈련 직후 아래 셀에서 바로 재생 가능
    - num_timesteps 기준으로 트리거 → n_envs가 몇이든 정확히 동작
    """

    def __init__(self, render_freq: int = 30_000, n_eval_episodes: int = 1,
                 video_dir: str = "training_videos"):
        super().__init__(verbose=0)
        self.render_freq = render_freq
        self.n_eval_episodes = n_eval_episodes
        self.video_dir = video_dir
        self.last_render_step = 0  # 마지막 저장 시점 추적
        os.makedirs(video_dir, exist_ok=True)

    def _on_step(self) -> bool:
        # 마지막 저장으로부터 render_freq 스텝 이상 지났을 때 실행
        # (n_calls 대신 num_timesteps 사용 → n_envs=16이어도 정확히 동작)
        if self.num_timesteps - self.last_render_step >= self.render_freq:
            self.last_render_step = self.num_timesteps
            print(f"\n🎬 [{self.num_timesteps:,} 스텝] 영상 저장 중...")

            try:
                # rgb_array 모드: 프레임을 numpy 배열로 받음 (창 안 띄움 → 커널 안전)
                render_env = gym.make("LunarLander-v2", render_mode="rgb_array")

                for ep in range(self.n_eval_episodes):
                    frames = []
                    obs, _ = render_env.reset()
                    done = False
                    total_reward = 0.0

                    while not done:
                        frames.append(render_env.render())  # 프레임 수집
                        action, _ = self.model.predict(obs, deterministic=True)
                        obs, reward, terminated, truncated, _ = render_env.step(action)
                        total_reward += reward
                        done = terminated or truncated

                    # mp4 파일로 저장 (파일명에 스텝 수 포함)
                    video_path = f"{self.video_dir}/step_{self.num_timesteps:07d}_ep{ep+1}.mp4"
                    imageio.mimsave(video_path, frames, fps=30)
                    print(f"   ✅ 저장 완료: {video_path}  (보상: {total_reward:.1f})")

                render_env.close()
                print(f"   다음 저장: {self.last_render_step + self.render_freq:,} 스텝")

            except Exception as e:
                print(f"   ⚠️ 영상 저장 실패: {e}")

        return True


# ✏️ render_freq 조절:
#   30_000 → 3만 스텝마다 저장 (자주 확인 가능, 영상 파일 많아짐)
#   50_000 → 5만 스텝마다 저장
render_callback = VideoRenderCallback(
    render_freq=100_000,
    n_eval_episodes=1,
    video_dir=VIDEO_DIR  # Google Drive 내 training_videos 폴더에 저장
)

### 6-4. imageio 설치

mp4 저장에 필요한 패키지입니다. 최초 1회만 실행하면 됩니다.

In [16]:
!pip install imageio imageio-ffmpeg

### 6-5. 모델 훈련 및 저장

훈련을 시작합니다. 3만 스텝마다 `training_videos/` 폴더에 mp4가 자동 저장됩니다.

| 훈련 단계 | 예상 보상 | 영상에서 보이는 것 |
|---|---|---|
| 0 ~ 10만 스텝 | 음수 ~ 0 | 충돌, 제자리 맴돔 |
| 10만 ~ 50만 스텝 | 0 ~ 100 | 어설프게 착륙 시도 |
| 50만 ~ 100만 스텝 | 100 ~ 200+ | 안정적으로 착륙 |

In [17]:
# 훈련 실행 (3만 스텝마다 training_videos/ 에 mp4 자동 저장)
model.learn(
    total_timesteps=800_000,
    callback=render_callback
)

# 모델 저장
model_name = "ppo-LunarLander-v2"
model_path = f"{MODEL_DIR}/{model_name}"
model.save(model_path)
print(f"\n✅ 훈련 완료! 모델이 '{model_path}.zip'으로 저장되었습니다.")

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 92.2     |
|    ep_rew_mean     | -195     |
| time/              |          |
|    fps             | 1416     |
|    iterations      | 1        |
|    time_elapsed    | 11       |
|    total_timesteps | 16384    |
---------------------------------
--------------------------------------------
| rollout/                |                |
|    ep_len_mean          | 90.2           |
|    ep_rew_mean          | -148           |
| time/                   |                |
|    fps                  | 1049           |
|    iterations           | 2              |
|    time_elapsed         | 31             |
|    total_timesteps      | 32768          |
| train/                  |                |
|    approx_kl            | 0.008492823    |
|    clip_fraction        | 0.0507         |
|    clip_range           | 0.2            |
|    entropy_loss         | -1.38          |
|    explained_variance   | -0

   ✅ 저장 완료: /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0100000_ep1.mp4  (보상: -224.6)
   다음 저장: 200,000 스텝
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 104           |
|    ep_rew_mean          | -76.6         |
| time/                   |               |
|    fps                  | 736           |
|    iterations           | 7             |
|    time_elapsed         | 155           |
|    total_timesteps      | 114688        |
| train/                  |               |
|    approx_kl            | 0.008023875   |
|    clip_fraction        | 0.0795        |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.31         |
|    explained_variance   | -0.0010921955 |
|    learning_rate        | 0.0003        |
|    loss                 | 185           |
|    n_updates            | 24            |
|    policy_gradient_loss | -0.00597      |
|    value_loss           | 467

   ✅ 저장 완료: /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0200000_ep1.mp4  (보상: -128.8)
   다음 저장: 300,000 스텝
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 126           |
|    ep_rew_mean          | -29.3         |
| time/                   |               |
|    fps                  | 638           |
|    iterations           | 13            |
|    time_elapsed         | 333           |
|    total_timesteps      | 212992        |
| train/                  |               |
|    approx_kl            | 0.003120171   |
|    clip_fraction        | 0.0464        |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.18         |
|    explained_variance   | 0.00015157461 |
|    learning_rate        | 0.0003        |
|    loss                 | 291           |
|    n_updates            | 48            |
|    policy_gradient_loss | -0.00162      |
|    value_loss           | 614

   ✅ 저장 완료: /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0300000_ep1.mp4  (보상: -346.4)
   다음 저장: 400,000 스텝
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 419          |
|    ep_rew_mean          | 3.78         |
| time/                   |              |
|    fps                  | 480          |
|    iterations           | 19           |
|    time_elapsed         | 647          |
|    total_timesteps      | 311296       |
| train/                  |              |
|    approx_kl            | 0.0053410972 |
|    clip_fraction        | 0.0284       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.13        |
|    explained_variance   | 0.44908452   |
|    learning_rate        | 0.0003       |
|    loss                 | 97.1         |
|    n_updates            | 72           |
|    policy_gradient_loss | -0.00156     |
|    value_loss           | 359          |
-------

   ✅ 저장 완료: /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0400000_ep1.mp4  (보상: -139.1)
   다음 저장: 500,000 스텝
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 732         |
|    ep_rew_mean          | 41.2        |
| time/                   |             |
|    fps                  | 398         |
|    iterations           | 25          |
|    time_elapsed         | 1028        |
|    total_timesteps      | 409600      |
| train/                  |             |
|    approx_kl            | 0.004553672 |
|    clip_fraction        | 0.0392      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.07       |
|    explained_variance   | 0.7088686   |
|    learning_rate        | 0.0003      |
|    loss                 | 67.4        |
|    n_updates            | 96          |
|    policy_gradient_loss | -0.00116    |
|    value_loss           | 129         |
---------------------------

   ✅ 저장 완료: /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0500000_ep1.mp4  (보상: -20.4)
   다음 저장: 600,000 스텝
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 945         |
|    ep_rew_mean          | 89.3        |
| time/                   |             |
|    fps                  | 343         |
|    iterations           | 31          |
|    time_elapsed         | 1479        |
|    total_timesteps      | 507904      |
| train/                  |             |
|    approx_kl            | 0.004818187 |
|    clip_fraction        | 0.0674      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.08       |
|    explained_variance   | 0.930887    |
|    learning_rate        | 0.0003      |
|    loss                 | 8.75        |
|    n_updates            | 120         |
|    policy_gradient_loss | -0.00167    |
|    value_loss           | 33.1        |
----------------------------

   ✅ 저장 완료: /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0600000_ep1.mp4  (보상: 155.4)
   다음 저장: 700,000 스텝
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 949         |
|    ep_rew_mean          | 104         |
| time/                   |             |
|    fps                  | 313         |
|    iterations           | 37          |
|    time_elapsed         | 1931        |
|    total_timesteps      | 606208      |
| train/                  |             |
|    approx_kl            | 0.010512825 |
|    clip_fraction        | 0.0639      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.01       |
|    explained_variance   | 0.909715    |
|    learning_rate        | 0.0003      |
|    loss                 | 25.4        |
|    n_updates            | 144         |
|    policy_gradient_loss | -0.00328    |
|    value_loss           | 68.1        |
----------------------------

   ✅ 저장 완료: /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0700000_ep1.mp4  (보상: 221.2)
   다음 저장: 800,000 스텝
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 895         |
|    ep_rew_mean          | 117         |
| time/                   |             |
|    fps                  | 311         |
|    iterations           | 43          |
|    time_elapsed         | 2260        |
|    total_timesteps      | 704512      |
| train/                  |             |
|    approx_kl            | 0.004946432 |
|    clip_fraction        | 0.0254      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.948      |
|    explained_variance   | 0.92395204  |
|    learning_rate        | 0.0003      |
|    loss                 | 62.8        |
|    n_updates            | 168         |
|    policy_gradient_loss | -0.0014     |
|    value_loss           | 77.7        |
----------------------------

   ✅ 저장 완료: /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0800000_ep1.mp4  (보상: 243.2)
   다음 저장: 900,000 스텝
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 884         |
|    ep_rew_mean          | 133         |
| time/                   |             |
|    fps                  | 325         |
|    iterations           | 49          |
|    time_elapsed         | 2467        |
|    total_timesteps      | 802816      |
| train/                  |             |
|    approx_kl            | 0.004706064 |
|    clip_fraction        | 0.0542      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.828      |
|    explained_variance   | 0.9680553   |
|    learning_rate        | 0.0003      |
|    loss                 | 3.67        |
|    n_updates            | 192         |
|    policy_gradient_loss | 0.000626    |
|    value_loss           | 36.2        |
----------------------------

### 6-6. 저장된 영상 확인

훈련 중 저장된 영상 목록을 확인하고, 원하는 시점의 영상을 노트북 안에서 바로 재생합니다.

In [18]:
import glob

# 저장된 영상 목록 출력
videos = sorted(glob.glob(f"{VIDEO_DIR}/*.mp4"))
print(f"총 {len(videos)}개의 영상이 저장되었습니다:\n")
for v in videos:
    print(" ", v)

총 8개의 영상이 저장되었습니다:

  /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0100000_ep1.mp4
  /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0200000_ep1.mp4
  /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0300000_ep1.mp4
  /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0400000_ep1.mp4
  /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0500000_ep1.mp4
  /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0600000_ep1.mp4
  /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0700000_ep1.mp4
  /content/drive/MyDrive/LunarLander/Certification/training_videos/step_0800000_ep1.mp4


In [19]:
# ✏️ 보고 싶은 영상 파일명을 아래에 입력하세요.
# 예) 초반: videos[0], 중반: videos[len(videos)//2], 최종: videos[-1]

# 모든 영상을 순서대로 재생
for video_path in videos:
    step = video_path.split("step_")[1].split("_")[0]  # 파일명에서 스텝 수 추출
    print(f"\n📽️  {int(step):,} 스텝 시점")
    display(Video(video_path, embed=True, width=400))


📽️  100,000 스텝 시점



📽️  200,000 스텝 시점



📽️  300,000 스텝 시점



📽️  400,000 스텝 시점



📽️  500,000 스텝 시점



📽️  600,000 스텝 시점



📽️  700,000 스텝 시점



📽️  800,000 스텝 시점


---
## 8. 모델 평가

`evaluate_policy`를 사용해 10번의 에피소드 동안 평균 보상을 측정합니다.

- **좋은 점수 기준**: 평균 보상 200점 이상이면 성공적인 착륙으로 간주됩니다.
- `deterministic=True`: 평가 시에는 확률적 선택 대신 가장 좋은 행동만 선택합니다.

In [20]:
# 평가용 환경 생성 (Monitor로 감싸서 보상 기록)
eval_env = Monitor(gym.make("LunarLander-v2", render_mode='rgb_array'))

# 10번 에피소드로 평균 보상 및 표준편차 계산
mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=10,
    deterministic=True
)

print(f"평균 보상: {mean_reward:.2f} ± {std_reward:.2f}")

평균 보상: 236.10 ± 22.80


---
## 9. Hugging Face Hub에 모델 업로드

훈련한 모델을 커뮤니티와 공유하려면 Hugging Face Hub에 업로드합니다.

### 사전 준비
1. [Hugging Face 계정 생성](https://huggingface.co/join)
2. [쓰기(write) 권한의 토큰 발급](https://huggingface.co/settings/tokens)
3. 아래 셀에서 로그인

### 8-1. Hugging Face 로그인

In [ ]:
from huggingface_hub import login

# ✏️ 본인의 HF Write 토큰으로 교체하세요 (https://huggingface.co/settings/tokens)
# ⚠️ 토큰은 절대 GitHub 등 외부에 공개하지 마세요!
login(token="YOUR_HF_TOKEN_HERE")

### 8-2. 모델 업로드

`repo_id`를 **본인의 HF 사용자명/저장소명** 형식으로 변경하세요.  
예: `"홍길동/ppo-LunarLander-v2"`

In [23]:
# ✏️ 아래 변수들을 본인 정보에 맞게 수정하세요.

env_id = "LunarLander-v2"           # 환경 이름
model_architecture = "PPO"           # 사용한 알고리즘
repo_id = "DitDahDitDit/ppo-LunarLander-v2"  # ← 본인의 HF ID로 변경!
commit_message = "Upload PPO LunarLander-v2 trained agent"

# 평가용 환경 (영상 녹화를 위해 rgb_array 모드 사용)
eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")])

# Hugging Face Hub에 모델 업로드
package_to_hub(
    model=model,
    model_name=model_name,
    model_architecture=model_architecture,
    env_id=env_id,
    eval_env=eval_env,
    repo_id=repo_id,
    commit_message=commit_message
)

print(f"✅ 모델이 https://huggingface.co/{repo_id} 에 업로드되었습니다!")

ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Saving video to /tmp/tmpaj2r0s3c/-step-0-to-step-1000.mp4
Moviepy - Building video /tmp/tmpaj2r0s3c/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmpaj2r0s3c/-step-0-to-step-1000.mp4



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Moviepy - Done !
Moviepy - video ready /tmp/tmpaj2r0s3c/-step-0-to-step-1000.mp4
ℹ Pushing repo DitDahDitDit/ppo-LunarLander-v2 to the Hugging Face
Hub


...

  /tmp/tmpuksqrsdp/replay.mp4 : 100%|##########|  175kB /  175kB            

  ...LunarLander-v2/policy.pth: 100%|##########| 44.0kB / 44.0kB            

  ...dp/ppo-LunarLander-v2.zip: 100%|##########|  149kB /  149kB            

  ...r-v2/policy.optimizer.pth: 100%|##########| 88.4kB / 88.4kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/DitDahDitDit/ppo-LunarLander-v2/tree/main/
✅ 모델이 https://huggingface.co/DitDahDitDit/ppo-LunarLander-v2 에 업로드되었습니다!
